# Xenium Data Preprocessing (MPP=0.5 기준)

Xenium 기술만 대상으로  
MPP=0.5 µm/pixel (20x) 기준 grid 패치 추출 + 2-head label 생성

In [ ]:
# ===== 1. 설정 및 라이브러리 =====
from glob import glob
import os
import numpy as np
import pandas as pd
import openslide
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')

def create_dir(path):
    os.makedirs(path, exist_ok=True)

# 하이퍼파라미터
TARGET_MPP = 0.5        # 20x 해상도 (µm/pixel)
CONTEXT_SCALE = 4       # 5x context = 4x wider
PATCH_SIZE = 512        # 패치 크기 (pixels)
MIN_TRANSCRIPTS = 10    # 패치 내 최소 transcript 수

# Spatial binning 파라미터 (Proportion 계산용)
N_BINS = 8              # 패치를 N_BINS × N_BINS sub-region으로 분할
BIN_SIZE = PATCH_SIZE // N_BINS  # 각 bin의 크기 (64 pixels ≈ 32µm at 20x)

# QV 필터링 threshold
QV_THRESHOLD = 20.0

# 대상 기술
TARGET_TECHS = ['Xenium']

# Marker genes (19개)
MARKER_GENES = [
    # Epithelial
    "EPCAM", "EGFR",
    # Fibroblast / Stromal
    "ACTA2", "PDGFRA", "PDGFRB", "SFRP4",
    # Endothelial
    "PECAM1",
    # Macrophage / Myeloid
    "CD68", "AIF1", "FCGR3A", "MRC1",
    # T cell
    "CD3E", "CD4", "CD8A", "TRAC",
    # B cell
    "CD79A", "MS4A1", "BANK1", "TCL1A"
]
NUM_GENES = len(MARKER_GENES)

# Gene alias dictionary
GENE_ALIAS = {
    "EPCAM":   ["TACSTD1"],
    "EGFR":    ["ERBB1"],
    "ACTA2":   ["SMA"],
    "PDGFRA":  ["CD140A"],
    "PDGFRB":  ["CD140B"],
    "PECAM1":  ["CD31"],
    "CD68":    ["GP110", "SCARD1"],
    "AIF1":    ["IBA1"],
    "FCGR3A":  ["CD16A", "CD16"],
    "MRC1":    ["CD206"],
    "CD3E":    ["T3E"],
    "CD8A":    ["CD8"],
    "TRAC":    ["TCRA", "TRA"],
    "CD79A":   ["IGA", "MB1"],
    "MS4A1":   ["CD20"],
    "TCL1A":   ["TCL1"],
}

def normalize_gene_name(g):
    """gene 이름을 정규화: 대문자, 특수문자 제거"""
    return str(g).strip().replace("-", "").replace("_", "").replace(" ", "").upper()

def resolve_gene_name(marker_gene, gene_names_in_data):
    """marker_gene의 primary name 또는 alias 중 데이터에 존재하는 이름을 반환."""
    if marker_gene in gene_names_in_data:
        return marker_gene
    if marker_gene in GENE_ALIAS:
        for alias in GENE_ALIAS[marker_gene]:
            if alias in gene_names_in_data:
                return alias
    norm_marker = normalize_gene_name(marker_gene)
    for g in gene_names_in_data:
        if normalize_gene_name(g) == norm_marker:
            return g
    if marker_gene in GENE_ALIAS:
        for alias in GENE_ALIAS[marker_gene]:
            norm_alias = normalize_gene_name(alias)
            for g in gene_names_in_data:
                if normalize_gene_name(g) == norm_alias:
                    return g
    return None

def load_parquet_transcripts(parquet_path, qv_threshold=QV_THRESHOLD):
    """Parquet 파일을 로드하고 QV 필터링 + feature_name 정규화 수행.
    Returns: DataFrame with columns [feature_name, he_x, he_y, qv, ...]
    """
    df = pd.read_parquet(parquet_path)

    # bytes → str 변환
    if len(df) > 0 and isinstance(df['feature_name'].iloc[0], bytes):
        df['feature_name'] = df['feature_name'].str.decode('utf-8')

    # BLANK / NegControl / antisense 프로브 제거
    df = df[~df['feature_name'].str.contains('BLANK|NegControl|antisense', case=False, na=False)]

    # QV 필터링
    df = df[df['qv'] >= qv_threshold]

    return df.reset_index(drop=True)

print(f'TARGET_MPP: {TARGET_MPP} µm/pixel')
print(f'PATCH_SIZE: {PATCH_SIZE}')
print(f'CONTEXT_SCALE: {CONTEXT_SCALE} (5x)')
print(f'N_BINS: {N_BINS} (spatial binning for proportion)')
print(f'BIN_SIZE: {BIN_SIZE} pixels ({BIN_SIZE * TARGET_MPP:.1f} µm)')
print(f'QV_THRESHOLD: {QV_THRESHOLD}')
print(f'MIN_TRANSCRIPTS: {MIN_TRANSCRIPTS}')
print(f'TARGET_TECHS: {TARGET_TECHS}')
print(f'MARKER_GENES ({NUM_GENES}): {MARKER_GENES}')

In [ ]:
# ===== 2. 데이터 경로 및 저장 디렉토리 생성 =====
data_path = '../../data/spatialTranscriptome/'
parquet_path = f'{data_path}transcripts/'  # parquet transcript 파일 경로
save_base = '../../data/marker_gene_spatial_expression'

meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')
print(f'Total slides in metadata: {len(meta_df)}')

# 대상 기술만 필터링
target_df = meta_df[meta_df['st_technology'].isin(TARGET_TECHS)].reset_index(drop=True)
print(f'Target slides ({TARGET_TECHS}): {len(target_df)}')
print(target_df['st_technology'].value_counts())

# parquet 파일 존재 여부 확인
parquet_available = []
for idx in range(len(target_df)):
    sid = target_df.iloc[idx]['id']
    pq_file = f'{parquet_path}{sid}_transcripts.parquet'
    parquet_available.append(os.path.exists(pq_file))
target_df['has_parquet'] = parquet_available
print(f'Slides with parquet: {sum(parquet_available)} / {len(target_df)}')

# organ 목록
organs = target_df['organ'].unique().tolist()
print(f'Organs: {organs}')

# 저장 디렉토리 생성
for o in organs:
    for t in TARGET_TECHS:
        create_dir(f'{save_base}/image/{t}/{o}')
        create_dir(f'{save_base}/image_5x/{t}/{o}')
        create_dir(f'{save_base}/label_proportion/{t}/{o}')
        create_dir(f'{save_base}/label_intensity/{t}/{o}')

print('Directories created.')


In [ ]:
# ===== Pass 1: Parquet 기반 전체 슬라이드에서 gene별 global stats 수집 =====
from tqdm import tqdm

gene_transcript_counts = {g: [] for g in MARKER_GENES}  # 슬라이드당 총 transcript 수
valid_slide_ids = []
slide_resolved_genes = {}
slide_gene_names_cache = {}  # 슬라이드별 고유 gene 목록 캐시

for idx in tqdm(range(len(target_df)), desc='Pass 1: Collecting gene stats (parquet)'):
    row = target_df.iloc[idx]
    sample_id = row['id']

    pq_file = f'{parquet_path}{sample_id}_transcripts.parquet'
    if not os.path.exists(pq_file):
        continue

    # feature_name만 먼저 로드해서 gene 매칭 확인
    df_names = pd.read_parquet(pq_file, columns=['feature_name'])
    if len(df_names) == 0:
        continue

    # bytes → str 변환
    if isinstance(df_names['feature_name'].iloc[0], bytes):
        df_names['feature_name'] = df_names['feature_name'].str.decode('utf-8')

    unique_genes = df_names['feature_name'].unique().tolist()
    slide_gene_names_cache[sample_id] = unique_genes

    # alias를 통해 실제 데이터 내 gene 이름 resolve
    resolved = {g: resolve_gene_name(g, unique_genes) for g in MARKER_GENES}
    missing = [g for g, r in resolved.items() if r is None]
    found_genes = [g for g in MARKER_GENES if resolved[g] is not None]

    if len(missing) > 0:
        print(f'  {sample_id}: missing {len(missing)} genes: {missing}')

    if len(found_genes) == 0:
        continue

    valid_slide_ids.append(sample_id)
    slide_resolved_genes[sample_id] = resolved

    # QV 필터 적용 후 marker gene transcript 수 수집
    df = load_parquet_transcripts(pq_file)

    for gene in found_genes:
        resolved_name = resolved[gene]
        count = (df['feature_name'] == resolved_name).sum()
        gene_transcript_counts[gene].append(count)

    # alias 매칭 알림
    aliased = {g: resolved[g] for g in found_genes if resolved[g] != g}
    if aliased:
        print(f'  {sample_id}: alias matched: {aliased}')

# gene별 통계 출력
print(f'\n{"Gene":>10s} | {"Slides":>6s} | {"Mean Count":>12s} | {"Median":>10s} | {"Min":>8s} | {"Max":>10s}')
print('-' * 70)
genes_with_no_data = []
for gene in MARKER_GENES:
    counts = gene_transcript_counts[gene]
    if len(counts) == 0:
        print(f'{gene:>10s} | *** NO DATA FOUND across all slides ***')
        genes_with_no_data.append(gene)
        continue
    arr = np.array(counts)
    print(f'{gene:>10s} | {len(arr):>6d} | {arr.mean():>12.1f} | {np.median(arr):>10.1f} | {arr.min():>8d} | {arr.max():>10d}')

if genes_with_no_data:
    print(f'\n⚠ Warning: {len(genes_with_no_data)} genes had no data: {genes_with_no_data}')

print(f'\nValid slides: {len(valid_slide_ids)} / {len(target_df)}')
print(f'QV threshold: {QV_THRESHOLD}')


In [ ]:
# ===== 3-1. 전체 슬라이드 gene 매칭 결과 확인 (Parquet 기반) =====
print(f'{"ID":>10s} | {"Tech":>15s} | {"Organ":>12s} | {"Total Genes":>11s} | {"Matched":>7s} | {"Alias":>5s} | {"Missing":>7s} | Details')
print('-' * 130)

alias_count = 0
missing_summary = {}

for idx in range(len(target_df)):
    row = target_df.iloc[idx]
    sample_id = row['id']
    st_tech = row['st_technology']
    organ_name = row['organ']

    pq_file = f'{parquet_path}{sample_id}_transcripts.parquet'
    if not os.path.exists(pq_file):
        print(f'{sample_id:>10s} | {"":>15s} | {"":>12s} | {"PARQUET NOT FOUND":>11s} |')
        continue

    # 캐시된 gene 이름 사용
    if sample_id in slide_gene_names_cache:
        unique_genes = slide_gene_names_cache[sample_id]
    else:
        df_names = pd.read_parquet(pq_file, columns=['feature_name'])
        if isinstance(df_names['feature_name'].iloc[0], bytes):
            df_names['feature_name'] = df_names['feature_name'].str.decode('utf-8')
        unique_genes = df_names['feature_name'].unique().tolist()

    resolved = {g: resolve_gene_name(g, unique_genes) for g in MARKER_GENES}
    matched = [g for g, r in resolved.items() if r is not None]
    aliased = {g: resolved[g] for g in MARKER_GENES if resolved[g] is not None and resolved[g] != g}
    missing = [g for g, r in resolved.items() if r is None]

    if aliased:
        alias_count += 1

    for mg in missing:
        missing_summary[mg] = missing_summary.get(mg, 0) + 1

    detail_parts = []
    if aliased:
        detail_parts.append(f'alias: {aliased}')
    if missing:
        detail_parts.append(f'missing: {missing}')
    detail_str = ' | '.join(detail_parts) if detail_parts else 'all OK'

    print(f'{sample_id:>10s} | {st_tech:>15s} | {organ_name:>12s} | {len(unique_genes):>11d} | {len(matched):>2d}/{NUM_GENES:<2d}  | {len(aliased):>5d} | {len(missing):>7d} | {detail_str}')

print('\n' + '=' * 80)
print(f'Total slides: {len(target_df)}')
print(f'Slides with alias match: {alias_count}')
print(f'\nMissing gene frequency (across all slides):')
for gene, cnt in sorted(missing_summary.items(), key=lambda x: -x[1]):
    print(f'  {gene:>10s}: {cnt} slides')


In [ ]:
# ===== 4. Grid 패치 추출 + 2-head label (Parquet 기반, QV 필터링, Spatial Binning) =====
from concurrent.futures import ThreadPoolExecutor
import glob as glob_module

save_image_dir = f'{save_base}/image'
save_image_5x_dir = f'{save_base}/image_5x'
save_proportion_dir = f'{save_base}/label_proportion'
save_intensity_dir = f'{save_base}/label_intensity'

# 이미 처리된 슬라이드 스킵
existing_files = set()
for f in glob_module.glob(f'{save_image_dir}/**/*.tiff', recursive=True):
    basename = os.path.splitext(os.path.basename(f))[0]
    parts = basename.rsplit('_', 2)
    if len(parts) == 3:
        existing_files.add(parts[0])
print(f'Already processed slides: {len(existing_files)}')

skipped = []
processed = []

for idx in tqdm(range(len(target_df)), desc='Grid extraction (Parquet + QV≥20)'):
    row = target_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']
    patch_count = 0

    if sample_id in existing_files:
        continue

    # Parquet 로드
    pq_file = f'{parquet_path}{sample_id}_transcripts.parquet'
    if not os.path.exists(pq_file):
        skipped.append((sample_id, 'parquet not found'))
        continue

    df = load_parquet_transcripts(pq_file)  # QV 필터링 + bytes/str 처리 + BLANK 제거
    if len(df) == 0:
        skipped.append((sample_id, 'no transcripts after QV filtering'))
        continue

    # 고유 gene 이름
    unique_genes = df['feature_name'].unique().tolist()

    # alias를 통해 실제 데이터 내 gene 이름 resolve
    resolved = {g: resolve_gene_name(g, unique_genes) for g in MARKER_GENES}
    missing = [g for g, r in resolved.items() if r is None]
    if len(missing) >= 1:
        skipped.append((sample_id, f'missing genes ({len(missing)}): {missing}'))
        continue

    # alias 매칭 알림
    aliased = {g: resolved[g] for g in MARKER_GENES if resolved[g] is not None and resolved[g] != g}
    if aliased:
        print(f'  {sample_id}: alias matched: {aliased}')

    # he_x, he_y 유효성 확인
    if df[['he_x', 'he_y']].isna().any().any():
        n_nan = df[['he_x', 'he_y']].isna().any(axis=1).sum()
        df = df.dropna(subset=['he_x', 'he_y']).reset_index(drop=True)
        print(f'  {sample_id}: dropped {n_nan} transcripts with NaN he_x/he_y')

    # marker gene transcript만 필터 (전체 transcript 카운트는 별도 계산)
    resolved_to_marker = {resolved[g]: g for g in MARKER_GENES}
    all_resolved_names = set(resolved_to_marker.keys())

    # WSI 로드
    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    # MPP
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        skipped.append((sample_id, 'MPP not found in image header'))
        slide.close()
        continue
    if slide_mpp < 0.1:
        slide_mpp *= 10

    downsample_20x = TARGET_MPP / slide_mpp
    downsample_5x = downsample_20x * CONTEXT_SCALE

    full_patch_20x = int(PATCH_SIZE * downsample_20x)
    full_patch_5x = int(PATCH_SIZE * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)
    n_patches_x = target_w // PATCH_SIZE
    n_patches_y = target_h // PATCH_SIZE

    if n_patches_x == 0 or n_patches_y == 0:
        skipped.append((sample_id, f'slide too small: {slide_w}x{slide_h}, mpp={slide_mpp:.4f}'))
        slide.close()
        continue

    # transcript 좌표를 20x pixel 좌표로 변환
    # he_x, he_y는 level0 pixel 좌표
    tx_20x = df['he_x'].values / downsample_20x
    ty_20x = df['he_y'].values / downsample_20x
    tx_genes = df['feature_name'].values

    def save_patch(px, py):
        patch_x0 = px * PATCH_SIZE
        patch_y0 = py * PATCH_SIZE

        # 패치 내 모든 transcript 찾기
        in_patch = (
            (tx_20x >= patch_x0) & (tx_20x < patch_x0 + PATCH_SIZE) &
            (ty_20x >= patch_y0) & (ty_20x < patch_y0 + PATCH_SIZE)
        )
        n_total = in_patch.sum()
        if n_total < MIN_TRANSCRIPTS:
            return False

        patch_tx = tx_20x[in_patch] - patch_x0  # 패치 내 로컬 좌표
        patch_ty = ty_20x[in_patch] - patch_y0
        patch_genes = tx_genes[in_patch]

        # Head A: Proportion (spatial binning 기반 spatial coverage)
        # 패치를 N_BINS × N_BINS sub-region으로 분할
        # 각 bin에서 해당 gene의 transcript가 1개 이상이면 양성
        # proportion = (양성 bin 수) / (전체 bin 수)
        bin_x = np.clip((patch_tx / BIN_SIZE).astype(int), 0, N_BINS - 1)
        bin_y = np.clip((patch_ty / BIN_SIZE).astype(int), 0, N_BINS - 1)
        bin_idx = bin_y * N_BINS + bin_x  # 1D bin index
        total_bins = N_BINS * N_BINS

        label_prop = np.zeros(NUM_GENES, dtype=np.float32)
        gene_counts = np.zeros(NUM_GENES, dtype=np.float32)
        for gi, marker in enumerate(MARKER_GENES):
            resolved_name = resolved[marker]
            gene_mask = (patch_genes == resolved_name)
            gene_counts[gi] = gene_mask.sum()
            if gene_counts[gi] > 0:
                # 해당 gene이 존재하는 고유 bin 수 계산
                positive_bins = len(np.unique(bin_idx[gene_mask]))
                label_prop[gi] = positive_bins / total_bins

        # Head B: Intensity (log transcript count, normalized)
        # = clip(log1p(gene transcript count), 0, 10) / 10
        label_int = np.clip(np.log1p(gene_counts), 0, 10.0) / 10.0

        # 20x patch
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        # 5x context patch
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = int(np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x)))
        y5 = int(np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x)))
        img_region_5x = slide.read_region((x5, y5), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        np.save(f'{save_proportion_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_prop)
        np.save(f'{save_intensity_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_int)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | MPP={slide_mpp:.4f} | ds={downsample_20x:.2f} | {patch_count} patches | {len(df)} transcripts (QV≥{QV_THRESHOLD})')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')

In [ ]:
# ===== 5. 처리 결과 요약 =====
if processed:
    total_patches = sum(cnt for _, cnt in processed)
    print(f'Total processed slides: {len(processed)}')
    print(f'Total patches saved: {total_patches}')
    print(f'Average patches per slide: {total_patches / len(processed):.1f}')
    print(f'\nTop 10 slides by patch count:')
    for s_id, cnt in sorted(processed, key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {s_id}: {cnt} patches')

for tech in TARGET_TECHS:
    n_files = len(glob(f'{save_base}/image/{tech}/**/*.tiff', recursive=True))
    print(f'\n{tech}: {n_files} image patches saved')

In [ ]:
# ===== 6. 시각화: 샘플 슬라이드 확인 (Parquet 기반, Spatial Binning) =====
from matplotlib.colors import Normalize
from matplotlib import cm

def visualize_slide_parquet(meta_row, data_path, parquet_dir, marker_genes,
                            target_mpp=0.5, patch_size=512, thumb_long_side=2000,
                            qv_threshold=QV_THRESHOLD, n_bins=N_BINS):
    sample_id = meta_row['id']
    image_file = meta_row['image_filename']
    bin_size = patch_size // n_bins

    # Parquet 로드 (QV 필터링 적용)
    pq_file = f'{parquet_dir}{sample_id}_transcripts.parquet'
    df = load_parquet_transcripts(pq_file, qv_threshold=qv_threshold)
    unique_genes = df['feature_name'].unique().tolist()

    # gene resolve
    resolved = {g: resolve_gene_name(g, unique_genes) for g in marker_genes}

    slide = openslide.open_slide(f'{data_path}wsis/{image_file}')
    slide_w, slide_h = slide.dimensions

    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        native_mag = float(str(meta_row['magnification']).replace('x', '').replace('X', ''))
        slide_mpp = 10.0 / native_mag
    if slide_mpp < 0.1:
        slide_mpp *= 10
    downsample_20x = target_mpp / slide_mpp

    aspect = slide_w / slide_h
    if aspect >= 1:
        tw, th = thumb_long_side, int(thumb_long_side / aspect)
    else:
        tw, th = int(thumb_long_side * aspect), thumb_long_side
    thumbnail = slide.get_thumbnail((tw, th))
    tw, th = thumbnail.size
    scale_x = tw / slide_w
    scale_y = th / slide_h

    # transcript 좌표 (he_x, he_y는 level0 pixel)
    he_x = df['he_x'].values
    he_y = df['he_y'].values
    tx_genes = df['feature_name'].values

    tx_20x = he_x / downsample_20x
    ty_20x = he_y / downsample_20x
    full_patch_20x = int(patch_size * downsample_20x)
    n_patches_x = int(slide_w / downsample_20x) // patch_size
    n_patches_y = int(slide_h / downsample_20x) // patch_size

    # 패치별 정보 수집
    patch_info = {}
    for py in range(n_patches_y):
        for px in range(n_patches_x):
            x0, y0 = px * patch_size, py * patch_size
            in_patch = (
                (tx_20x >= x0) & (tx_20x < x0 + patch_size) &
                (ty_20x >= y0) & (ty_20x < y0 + patch_size)
            )
            cnt = int(in_patch.sum())
            if cnt >= MIN_TRANSCRIPTS:
                patch_info[(px, py)] = (cnt, in_patch)

    genes_to_show = ['EPCAM', 'CD68', 'CD3E']
    cmaps_show = ['Reds', 'Blues', 'Greens']

    fig, axes = plt.subplots(3, 3, figsize=(28, 26))

    # Row 0, Col 0: Grid + Transcripts
    ax = axes[0, 0]
    ax.imshow(thumbnail)
    for px_line in range(n_patches_x + 1):
        ax.axvline(x=px_line * full_patch_20x * scale_x, color='cyan', linewidth=0.4, alpha=0.4)
    for py_line in range(n_patches_y + 1):
        ax.axhline(y=py_line * full_patch_20x * scale_y, color='cyan', linewidth=0.4, alpha=0.4)
    spot_size = max(0.5, min(5, thumb_long_side / 800))
    if patch_info:
        max_cnt = max(cnt for cnt, _ in patch_info.values())
        norm_cnt = Normalize(vmin=1, vmax=max(max_cnt, 2))
        cmap_patch = cm.YlGn
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.3,
                                 edgecolor='lime', linewidth=0.8)
            ax.add_patch(rect)
    # transcript 위치 표시 (subsampled)
    subsample = max(1, len(he_x) // 50000)
    ax.scatter(he_x[::subsample] * scale_x, he_y[::subsample] * scale_y,
               s=spot_size, c='red', alpha=0.3, edgecolors='none')
    ax.set_title(f'Grid + Transcripts | {n_patches_x}x{n_patches_y} grid | '
                 f'{len(patch_info)} valid | {len(df)} transcripts (QV≥{qv_threshold}) | MPP={slide_mpp:.4f}',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

    # Row 0, Col 1: Transcripts per Patch
    ax = axes[0, 1]
    ax.imshow(thumbnail, alpha=0.4)
    if patch_info:
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_patch, norm=norm_cnt)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='transcripts per patch')
    ax.set_title('Transcripts per Patch', fontsize=12, fontweight='bold')
    ax.axis('off')

    # Row 0, Col 2: Summary stats
    ax = axes[0, 2]
    ax.axis('off')
    if patch_info:
        tx_counts = [cnt for cnt, _ in patch_info.values()]
        stats_text = (
            f'Total transcripts (QV≥{qv_threshold}): {len(df):,}\n'
            f'Valid patches: {len(patch_info)} / {n_patches_x * n_patches_y}\n'
            f'Transcripts/patch: min={min(tx_counts)}, med={int(np.median(tx_counts))}, '
            f'max={max(tx_counts)}, mean={np.mean(tx_counts):.0f}\n\n'
            f'Technology: {meta_row["st_technology"]}\n'
            f'Organ: {meta_row["organ"]}\n'
            f'MPP: {slide_mpp:.4f} um/pixel\n'
            f'Downsample: {downsample_20x:.2f}x\n'
            f'QV threshold: {qv_threshold}\n'
            f'Spatial binning: {n_bins}x{n_bins} ({bin_size}px per bin)'
        )
        ax.text(0.1, 0.5, stats_text, transform=ax.transAxes,
                fontsize=14, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_title('Summary', fontsize=12, fontweight='bold')

    # Row 1: Proportion (spatial binning 기반 spatial coverage)
    total_bins = n_bins * n_bins
    norm_prop = Normalize(vmin=0, vmax=1.0)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[1, col]
        ax.imshow(thumbnail, alpha=0.4)
        resolved_name = resolved.get(gene)
        if resolved_name is None:
            ax.set_title(f'{gene} -- NOT FOUND', fontsize=12)
            ax.axis('off')
            continue
        for (px, py), (cnt, in_patch) in patch_info.items():
            x0, y0 = px * patch_size, py * patch_size
            patch_tx = tx_20x[in_patch] - x0
            patch_ty = ty_20x[in_patch] - y0
            patch_genes_local = tx_genes[in_patch]

            gene_mask = (patch_genes_local == resolved_name)
            if gene_mask.sum() > 0:
                bx = np.clip((patch_tx[gene_mask] / bin_size).astype(int), 0, n_bins - 1)
                by = np.clip((patch_ty[gene_mask] / bin_size).astype(int), 0, n_bins - 1)
                b_idx = by * n_bins + bx
                prop = len(np.unique(b_idx)) / total_bins
            else:
                prop = 0.0

            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_prop(prop))
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_prop)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='spatial coverage')
        ax.set_title(f'{gene} -- Proportion (spatial coverage, {n_bins}x{n_bins} bins)', fontsize=12, fontweight='bold')
        ax.axis('off')

    # Row 2: Intensity (log transcript count, normalized)
    norm_int = Normalize(vmin=0, vmax=1)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[2, col]
        ax.imshow(thumbnail, alpha=0.4)
        resolved_name = resolved.get(gene)
        if resolved_name is None:
            ax.set_title(f'{gene} -- NOT FOUND', fontsize=12)
            ax.axis('off')
            continue
        for (px, py), (cnt, in_patch) in patch_info.items():
            patch_genes_local = tx_genes[in_patch]
            gene_cnt = (patch_genes_local == resolved_name).sum()
            intensity = np.clip(np.log1p(gene_cnt), 0, 10.0) / 10.0
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_int(intensity))
            rect = plt.Rectangle((rx, ry), rw, rh, fill=True,
                                 facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_int)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='intensity (norm)')
        ax.set_title(f'{gene} -- Intensity (log1p count / 10)', fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'{sample_id}  |  {meta_row["st_technology"]} / {meta_row["organ"]}  |  QV≥{qv_threshold}  |  bins={n_bins}x{n_bins}',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
    slide.close()


visualize_slide_parquet(target_df.iloc[10], data_path, parquet_path, MARKER_GENES,
                        target_mpp=TARGET_MPP, patch_size=PATCH_SIZE)